In [110]:
# importando as bibliotecas necessárias
import pandas as pd
import numpy as np
import datetime as date

In [111]:
# importando o dataset csv
path = 'dataset.csv'
df = pd.read_csv(path)
df

,Nome,Nasc.,Pos.,Gols,GP,Ast.,Des.,Int.,Def.,PD,Part.,Valor
0,Lamine Yamal,13/07/2007,ATA,18,-,15,-,-,-,-,115,€200.00m
1,Erling Haaland,21/07/2000,ATA,307,55,65,-,-,-,-,387,€200.00m
2,Kylian Mbappé,20/12/1998,ATA,385,49,142,-,-,-,-,488,€200.00m
3,Vinicius Junior,12/07/2000,ATA,137,-,111,-,-,-,-,455,€150.00m
4,Bukayo Saka,05/09/2001,ATA,78,12,64,-,-,-,-,312,€130.00m
...,...,...,...,...,...,...,...,...,...,...,...,...
95,Danny Welbeck,26/11/1990,ATA,92,-,42,-,-,-,-,452,€4.00m
96,Jean-Philippe Mateta,28/06/1997,ATA,68,5,12,-,-,-,-,215,€35.00m
97,Adam Wharton,06/02/2004,MEI,5,-,14,54*,32*,-,-,98,€60.00m
98,Kobbie Mainoo,19/04/2005,MEI,6,-,8,-,-,-,-,74,€40.00m


Verificando os tipos das colunas

In [141]:
df.dtypes

Nome     string[python]
Nasc.    datetime64[ns]
Idade             int64
Pos.     string[python]
PT                int64
Gols              Int64
GP                Int64
Ast.              Int64
Des.              Int64
Int.              Int64
Def.              Int64
PD                Int64
Part.             int64
Valor           float64
dtype: object

Vou ter que alterar os tipos das colunas para que isso funcione corretamente

In [113]:
# percebi que existem alguns valores com * depois dos numeros, irei remover esses *
df['Gols'] = df['Gols'].str.replace('*', '', regex=False)
df['GP'] = df['GP'].str.replace('*', '', regex=False)
df['Ast.'] = df['Ast.'].str.replace('*', '', regex=False)
df['Des.'] = df['Des.'].str.replace('*', '', regex=False)
df['Int.'] = df['Int.'].str.replace('*', '', regex=False)
df['Def.'] = df['Def.'].str.replace('*', '', regex=False)
df['PD'] = df['PD'].str.replace('*', '', regex=False)

In [114]:
# alterando o tipo das colunas
df['Nome'] = df['Nome'].astype('string')

# formatando a coluna de data
df['Nasc.'] = pd.to_datetime(df['Nasc.'], format='%d/%m/%Y', errors='coerce')

# criando uma coluna idade
df['Idade'] = (date.datetime.now() - df['Nasc.']).apply(lambda x: x.days // 365)

# movendo a coluna Idade para depois da coluna Nasc.
cols = df.columns.tolist()
cols.insert(cols.index('Nasc.') + 1, cols.pop(cols.index('Idade')))
df = df[cols]

# transformando as posições em strings
df['Pos.'] = df['Pos.'].astype('string')

# mapeando as posições
position_mapping = {
    'GOL': 0,
    'DEF': 1,
    'MEI': 2,
    'ATA': 3
}

# criando uma nova coluna para transformar as posições em numeros categoricos
df['PT'] = df['Pos.'].map(position_mapping)

# movendo a coluna PT para depois da coluna Pos.
cols = df.columns.tolist()
cols.insert(cols.index('Pos.') + 1, cols.pop(cols.index('PT')))
df = df[cols]

# Agora irei substituir esses ifens por NaN
df.replace('-', np.nan, inplace=True)

# agora eu quero transformar as colunas numericas que estao como object em int
numeric_columns = ['Gols', 'GP', 'Ast.', 'Des.', 'Int.', 'Def.', 'PD']
df[numeric_columns] = df[numeric_columns].astype('Int64')

In [136]:
df.dtypes

Nome     string[python]
Nasc.    datetime64[ns]
Idade             int64
Pos.     string[python]
PT                int64
Gols              Int64
GP                Int64
Ast.              Int64
Des.              Int64
Int.              Int64
Def.              Int64
PD                Int64
Part.             int64
Valor            object
dtype: object

In [140]:
df.head()

,Nome,Nasc.,Idade,Pos.,PT,Gols,GP,Ast.,Des.,Int.,Def.,PD,Part.,Valor
0,Lamine Yamal,2007-07-13,18,ATA,3,18,2,15,0,0,0,0,115,200000000.0
1,Erling Haaland,2000-07-21,25,ATA,3,307,55,65,0,0,0,0,387,200000000.0
2,Kylian Mbappé,1998-12-20,27,ATA,3,385,49,142,0,0,0,0,488,200000000.0
3,Vinicius Junior,2000-07-12,25,ATA,3,137,18,111,0,0,0,0,455,150000000.0
4,Bukayo Saka,2001-09-05,24,ATA,3,78,12,64,0,0,0,0,312,130000000.0


Agora é importante que primeiramente eu lide com os valores faltantes dessa tabela

In [137]:
# Gols
df['Gols'] = df['Gols'].fillna(0)

# GP para jogadores sem gols
df.loc[df["GP"].isna() & (df["Gols"] == 0), "GP"] = 0

# GP pra quem nao é atacante
df.loc[df["GP"].isna() & (df["Pos."] != "ATA"), "GP"] = 0

# calculando a média de gols por penalti para atacantes
mean_goals_penalty = df[(df['Pos.'] == 'ATA') & (df['Gols'] > 0) & (df['GP'] > 0)].assign(pgp=lambda x: x['GP'] / x['Gols'])['pgp'].mean()
mean_goals_penalty 

# esse valor vai ser usado para preencher os vazios restantes de GP (atacantes com gols)
df["GP"] = df["GP"].fillna((df["Gols"] * mean_goals_penalty).round().astype('Int64'))

# Agora preenchendo as outras estatisticas menos importantes com as medias de cada posição
cols_stats = ['Ast.', 'Des.', 'Int.']
for col in cols_stats:
    # preenchendo com a média por posição
    df[col] = df[col].fillna(df.groupby('Pos.')[col].transform('mean').round().astype('Int64'))

    # preenchendo os valores restantes com 0 (ou seja, caso a coluna inteira esteja vazia)
    df[col] = df[col].fillna(0)

# se o jogador não é goleiro, defesas e Penaltis Defendidos são 0
df[df['Pos.'] != 'GOL'] = df[df['Pos.'] != 'GOL'].fillna({'Def.': 0, 'PD': 0})

# preenchendo as defesas dos goleiros com a média por partida
mean_defenses = df[(df['Pos.'] == 'GOL') & (df['Def.'] > 0)].assign(dpg=lambda x: x['Def.'] / x['Part.'])['dpg'].mean()
df['Def.'] = df['Def.'].fillna((df['Part.'] * mean_defenses).round()).astype('Int64')

# preenchendo os penaltis defendidos com a media por partida
mean_penalties_defended = df[(df['Pos.'] == 'GOL') & (df['PD'] > 0)].assign(pdpg=lambda x: x['PD'] / x['Part.'])['pdpg'].mean()
df['PD'] = df['PD'].fillna((df['Part.'] * mean_penalties_defended).round()).astype('Int64')

In [139]:
# agora formatando o valor do jogador para float
def format_value(value):
    # se for nulo, retorna nulo
    if pd.isna(value):
        return None

    # remove o símbolo de euro e converte para minúsculas
    value = str(value).replace('€', '').lower()
    if value.endswith('m'):
        return float(value[:-1]) * 1_000_000
    else:
        return float(value)

# aplicando na coluna Valor
df['Valor'] = df['Valor'].apply(format_value)

Dados finalmente formatados, agora irei começar a separar e preparar o treinamento e teste para previsão de valores de outros jogadores

In [144]:
new_df = df.copy()
new_df = new_df[['Idade', 'PT', 'Part.', 'Gols', 'GP', 'Ast.', 'Des.', 'Int.', 'Def.', 'PD', 'Valor']]
new_df

,Idade,PT,Part.,Gols,GP,Ast.,Des.,Int.,Def.,PD,Valor
0,18,3,115,18,2,15,0,0,0,0,200000000.0
1,25,3,387,307,55,65,0,0,0,0,200000000.0
2,27,3,488,385,49,142,0,0,0,0,200000000.0
3,25,3,455,137,18,111,0,0,0,0,150000000.0
4,24,3,312,78,12,64,0,0,0,0,130000000.0
...,...,...,...,...,...,...,...,...,...,...,...
95,35,3,452,92,12,42,0,0,0,0,4000000.0
96,28,3,215,68,5,12,0,0,0,0,35000000.0
97,21,2,98,5,0,14,54,32,0,0,60000000.0
98,20,2,74,6,0,8,59,30,0,0,40000000.0


In [145]:
# importando bibliotecas para treinamento e teste
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

In [146]:
# pegando todas as colunas menos a coluna Valor como features
X = new_df.drop('Valor', axis=1)

# pegando a coluna Valor como target
y = new_df['Valor']

In [147]:
# separando treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Regressão Linear
model = LinearRegression()
# treinando o modelo
model.fit(X_train, y_train)
# fazendo previsões
predict = model.predict(X_test)
# calculando o R2
r2 = r2_score(y_test, predict)
r2

0.1919734511575567

Valor muito baixo para regressão linear

In [154]:
# Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
# treinando o modelo
model.fit(X_train, y_train)
# fazendo previsões
predict = model.predict(X_test)
# calculando o R2
r2 = r2_score(y_test, predict)
r2

0.21187798641990507

Valor também baixo para o random forest

In [155]:
# XGBoost Regressor
model = XGBRegressor(n_estimators=100, random_state=42)
# treinando o modelo
model.fit(X_train, y_train)
# fazendo previsões
predict = model.predict(X_test)
# calculando o R2
r2 = r2_score(y_test, predict)
r2

0.4206761370297448

Foi o melhor resultado, mas longe de ser satisfatorio